In [ ]:
import glob
import pandas as pd
import tqdm

base_path = "/srv/nfs-data/sisko/storage/kamitani_sound/ds006319"
events_files = glob.glob(f"{base_path}/sub-*/ses-*/func/*_events.tsv")

def parse_stimulus_name(stimulus_name: str):
    """
    Parsing robusto per:
      vggtest_jPKhBkLgFLk_16000_26000
      vggtest_-_f_ClsDxyc_80000_90000
    Regola:
      - ultimi due token: start_ms, end_ms
      - youtube_id = "_".join(parts[1:-2])
    """
    s = str(stimulus_name)
    parts = s.split("_")
    if len(parts) < 4:
        return None

    try:
        start_ms = int(parts[-2])
        end_ms = int(parts[-1])
    except ValueError:
        return None

    youtube_id = "_".join(parts[1:-2])

    # YouTube ID standard: 11 char
    if len(youtube_id) != 11:
        return None

    return {
        "youtube_id": youtube_id,
        "start_ms": start_ms,
        "end_ms": end_ms,
        "start_sec": start_ms / 1000.0,
        "end_sec": end_ms / 1000.0,
        "duration_sec": (end_ms - start_ms) / 1000.0,
    }

rows = []
bad = []

for f in tqdm.tqdm(events_files):
    df = pd.read_csv(f, sep="\t")
    df = df[(df["trial_type"] == 1) & (df["stimulus_name"].notna())]

    for s in df["stimulus_name"].unique():
        parsed = parse_stimulus_name(s)
        print(str(s))
        if parsed is None:
            bad.append({"stimulus_name": str(s), "events_file": f})
            continue

        rows.append({
            "stimulus_name": str(s),
            "youtube_id": parsed["youtube_id"],
            "start_ms": parsed["start_ms"],
            "end_ms": parsed["end_ms"],
            "start_sec": parsed["start_sec"],
            "end_sec": parsed["end_sec"],
            "duration_sec": parsed["duration_sec"],
        })

manifest = pd.DataFrame(rows).drop_duplicates("stimulus_name").sort_values("stimulus_name").reset_index(drop=True)
print("Unique stimuli parsed:", len(manifest))

manifest.to_csv("stimuli_manifest.csv", index=False)
print("Saved: stimuli_manifest.csv")



In [2]:
manifest = pd.DataFrame(rows).drop_duplicates("stimulus_name").sort_values("stimulus_name")
print("Unique stimuli:", len(manifest))
# manifest.to_csv("stimuli_manifest.csv", index=False)

Unique stimuli: 1331


In [ ]:
import os
import subprocess
import pandas as pd

manifest_path = "stimuli_manifest.csv"
out_dir = "/srv/nfs-data/sisko/storage/kamitani_sound/vggsound_wav"
os.makedirs(out_dir, exist_ok=True)

df = pd.read_csv(manifest_path)

def run_cmd(cmd):
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    return res.returncode, res.stdout, res.stderr

failed = []
for _, row in tqdm.tqdm(df.iterrows()):
    stim = row["stimulus_name"]
    ytid = row["youtube_id"]
    start = float(row["start_sec"])
    end = float(row["end_sec"])

    out_path = os.path.join(out_dir, f"{stim}.wav")
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        continue

    url = f"https://www.youtube.com/watch?v={ytid}"
    cmd = [
        "yt-dlp",
        "-x",
        "--audio-format", "wav",
        "--postprocessor-args", f"ffmpeg:-ss {start} -to {end}",
        "-o", out_path,
        url
    ]
    print("Saving to:", out_path)
    code, _, err = run_cmd(cmd)
    if code != 0 or not (os.path.exists(out_path) and os.path.getsize(out_path) > 0):
        failed.append({
            "stimulus_name": stim,
            "url": url,
            "returncode": code,
            "error": err[-800:]  
        })
        print(f"Failed to download {stim} from {url}, return code {code}, error: {err}")

print("Done.")
print("Failed:", len(failed))

if failed:
    pd.DataFrame(failed).to_csv("download_failed.csv", index=False)
    print("Saved failures to download_failed.csv")


In [12]:
import torchaudio
from IPython.display import Audio

wav_path = "/srv/nfs-data/sisko/storage/kamitani_sound/vggsound_wav/vggtest_2DyD824KPFM_10000_20000.wav"
waveform, sr = torchaudio.load(wav_path)

print("Waveform shape:", waveform.shape)  # (channels, samples)
print("Sample rate:", sr)

Audio(waveform.numpy(), rate=sr)


Waveform shape: torch.Size([2, 480000])
Sample rate: 48000
